# Method 2 — Cross-Encoder (Kaggle T4)

Phase 3 của `docs/method2_plan.md`: sinh nhãn, curriculum 2 giai đoạn, gate theo head. Ngân sách ~4.5h GPU.

**Ba quy tắc sống còn trên Kaggle** (§8 `docs/method2_plan.md`):

1. Bật **Save & Run All (Commit)** cho job dài — session tương tác bị ngắt sau ~20 phút không tương tác, commit run chạy nền đủ 12h.
2. Checkpoint mỗi 500 step vào `/kaggle/working`, và **luôn** hỗ trợ `resume_from`.
3. Cache model HuggingFace thành Kaggle Dataset (`BAAI/bge-m3` ~2.3GB) thay vì tải lại mỗi session.


In [1]:
# ===== Cell 0: dò dataset + HF cache =====
# PHẢI chạy trước mọi import transformers: thư viện chốt cache lúc import,
# set HF_HOME sau đó thì không còn tác dụng.
import os
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')


def _dirs_within(base: Path, max_depth: int = 4):
    """Mọi thư mục tới độ sâu `max_depth`, bỏ qua `hub/` cho nhanh."""
    frontier, seen = [base], []
    for _ in range(max_depth):
        nxt = []
        for d in frontier:
            try:
                children = [c for c in d.iterdir() if c.is_dir() and c.name != 'hub']
            except (PermissionError, OSError):
                continue
            seen.extend(children)
            nxt.extend(children)
        frontier = nxt
    return seen


def find_root(marker: str, label: str) -> Path:
    """Tìm thư mục chứa `marker`.

    Kaggle mount theo dạng /kaggle/input/datasets/<user>/<ds>/<ds>/, và số tầng
    đổi theo cách upload. Dò theo marker thì không phải hardcode username hay
    độ sâu — upload kiểu nào cũng tìm ra.
    """
    for d in [INPUT_ROOT] + _dirs_within(INPUT_ROOT):
        if (d / marker).exists():
            return d
    raise SystemExit(
        f'Không tìm thấy {label}: không thư mục nào dưới {INPUT_ROOT} có {marker}.\n'
        'Kiểm tra đã Add đủ 3 dataset ở sidebar Input chưa.'
    )


SRC_ROOT = find_root('src/models/preflight.py', 'dataset src')
DATA_ROOT = find_root('method2/manifest.json', 'dataset data')
HF_HOME = find_root('hub/models--BAAI--bge-m3', 'dataset hf-cache')

print('SRC :', SRC_ROOT)
print('DATA:', DATA_ROOT)
print('HF  :', HF_HOME)

os.environ['HF_HOME'] = str(HF_HOME)
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

# Có thư mục model chưa đủ — thiếu file trọng số thì lỗi chỉ lộ ra lúc nạp
# model, sau khi đã tốn thời gian cài đặt và copy.
for name in ('models--BAAI--bge-m3', 'models--xlm-roberta-base'):
    weights = [
        f for f in (HF_HOME / 'hub' / name).rglob('*')
        if f.is_file() and f.suffix in ('.safetensors', '.bin') and f.stat().st_size > 10**8
    ]
    assert weights, f'{name}: không có file trọng số > 100 MB'
    print(f'  {name}: {max(f.stat().st_size for f in weights) / 1024**3:.2f} GB')
print('\nHF cache OK')


SRC : /kaggle/input/datasets/dathq12/toolcalling-vi-src/toolcalling-vi-src
DATA: /kaggle/input/datasets/dathq12/toolcalling-vi-data/toolcalling-vi-data
HF  : /kaggle/input/datasets/dathq12/toolcalling-vi-hf-cache/toolcalling-vi-hf-cache
  models--BAAI--bge-m3: 2.12 GB
  models--xlm-roberta-base: 1.04 GB

HF cache OK


In [2]:
# ===== Cell 1: env — PIN version =====
# Ba package này quyết định API training VÀ tên metric của
# InformationRetrievalEvaluator. Đổi bản là đổi khoá metric, hỏng cả
# load_best_model_at_end lẫn khả năng so sánh giữa các run.
!pip install -q 'transformers==5.15.1' 'sentence-transformers==6.0.0' 'peft==0.20.0' \
                accelerate jsonschema rank_bm25 datasets

# PEFT 0.20 raise nếu image có torchao < 0.16. Method 2 không dùng
# torchao quantization nên gỡ hẳn là xong.
!pip uninstall -y -q torchao 2>/dev/null || true

# torch KHÔNG pin: Kaggle cài sẵn bản CUDA riêng, ép cài lại vừa chậm vừa
# dễ lệch CUDA runtime của image. Chỉ ghi nhận version vào manifest.
import torch

free, total = torch.cuda.mem_get_info()
n_gpu = torch.cuda.device_count()
print(torch.cuda.get_device_name(0), f'{free/1024**3:.1f} / {total/1024**3:.1f} GB free')
print('số GPU:', n_gpu)

# sentence-transformers tự bọc DataParallel khi thấy >1 GPU. Với GradCache
# gọi model hàng trăm lần mỗi step thì phí đồng bộ cộng dồn rất nhanh.
if n_gpu > 1:
    print('  >1 GPU — truyền --single-gpu cho MỌI lệnh train')

# T4 là Turing (sm_75), KHÔNG có bf16 phần cứng. torch vẫn có thể báo
# is_bf16_supported()=True vì hỗ trợ qua emulation, chậm hơn fp16.
# Giữ fp16 bất kể giá trị này.
print('bf16 (emulated trên T4, vẫn dùng fp16):', torch.cuda.is_bf16_supported())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.6/739.6 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 30.2 MB/s eta 0:00:00
Tesla T4 14.5 / 14.6 GB free
số GPU: 2
  >1 GPU — truyền --single-gpu cho MỌI lệnh train
bf16 (emulated trên T4, vẫn dùng fp16): True


In [3]:
# ===== Cell 2: copy code + data vào /kaggle/working =====
# Dataset chỉ đọc, mà code ghi checkpoint và dùng đường dẫn tương đối, nên
# phải copy sang thư mục ghi được. Dùng path đã dò ở Cell 0.
import shutil

WORK = Path('/kaggle/working')
# `scripts` cần thiết: benchmark_biencoder.py chạy trên Kaggle.
for name in ('src', 'configs', 'scripts'):
    target = WORK / name
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(SRC_ROOT / name, target)

# Dataset data bắt đầu thẳng bằng method2/ custom_vi/ benchmark_vi/ (KHÔNG có
# tầng `data/`), còn code tham chiếu `data/method2/...` → copy vào data/.
data_dir = WORK / 'data'
if data_dir.exists():
    shutil.rmtree(data_dir)
data_dir.mkdir(parents=True)
for child in DATA_ROOT.iterdir():
    dest = data_dir / child.name
    shutil.copytree(child, dest) if child.is_dir() else shutil.copy2(child, dest)

%cd /kaggle/working

import json, glob, sys
sys.path.insert(0, '/kaggle/working')
# HF_HOME đã set ở Cell 0, kế thừa sang mọi tiến trình con `!python`.

print('src    :', sorted(p.name for p in (WORK / 'src').iterdir()))
print('data   :', sorted(p.name for p in data_dir.iterdir()))


/kaggle/working
src    : ['README.md', 'data', 'evaluation', 'models']
data   : ['benchmark_vi', 'custom_vi', 'method2']


In [4]:
# ===== Cell 3: kiểm tra bản copy TRƯỚC khi preflight =====
# Preflight kiểm tra tính đúng đắn của dữ liệu; cell này kiểm tra bước copy —
# tách ra để khi hỏng thì biết ngay là hỏng ở đâu.
REQUIRED = [
    'data/method2/decontamination.json',
    'data/method2/manifest.json',
    'data/method2/tool_pool.json',
    'data/method2/biencoder/train.jsonl',
    'data/method2/biencoder/val.jsonl',
    'data/method2/biencoder/pairs_stats.json',
    'data/method2/crossencoder/train.jsonl',
    'data/method2/crossencoder/val.jsonl',
    'data/method2/label_stats.json',
    'data/custom_vi/v1/test_seen.jsonl',
    'data/benchmark_vi/test.jsonl',
    'configs/method2/biencoder.yaml',
    'configs/method2/crossencoder.yaml',
    'configs/method2/pinned_versions.json',
    'src/models/preflight.py',
]
missing = []
for rel in REQUIRED:
    path = WORK / rel
    if path.exists() and path.stat().st_size > 0:
        print(f'  {path.stat().st_size / 1024**2:8.2f} MB  {rel}')
    else:
        missing.append(rel)
        print(f'  {"THIẾU":>11}  {rel}')
assert not missing, f'Copy chưa đủ: {missing}'

# import được thì mới chắc src/ copy nguyên vẹn.
import importlib

importlib.import_module('src.models.preflight')
manifest = json.load(open('data/method2/manifest.json', encoding='utf-8'))
print('\nsnapshot commit:', manifest.get('git_commit'))
print('copy OK')


     10.81 MB  data/method2/decontamination.json
      0.00 MB  data/method2/manifest.json
      5.01 MB  data/method2/tool_pool.json
     46.20 MB  data/method2/biencoder/train.jsonl
      5.47 MB  data/method2/biencoder/val.jsonl
      0.00 MB  data/method2/biencoder/pairs_stats.json
     91.13 MB  data/method2/crossencoder/train.jsonl
     11.29 MB  data/method2/crossencoder/val.jsonl
      0.00 MB  data/method2/label_stats.json
      5.18 MB  data/custom_vi/v1/test_seen.jsonl
     14.18 MB  data/benchmark_vi/test.jsonl
      0.00 MB  configs/method2/biencoder.yaml
      0.00 MB  configs/method2/crossencoder.yaml
      0.00 MB  configs/method2/pinned_versions.json
      0.01 MB  src/models/preflight.py

snapshot commit: 6eb1b582b4eaba9ff8a9b51c4d88ae5bd0898465
copy OK


## Pre-flight — cổng fail-closed TRƯỚC mọi training

```
decontamination.json tồn tại
        ↓
SHA-256 == manifest.json
        ↓
overlap train/val/test == 0
        ↓
unseen positive leakage == 0
        ↓
package versions khớp bản đã pin
        ↓
CHO PHÉP TRAIN
```

Thiếu file hoặc hash lệch → job dừng ngay, **không rebuild tự động**. Nếu
experiment chính tự dựng lại index từ dữ liệu đang có trên máy thì ta mất
đúng thứ cần đảm bảo: bằng chứng model được train trên đúng split đã kiểm
định. Rebuild là lệnh preprocessing riêng, chạy ở local rồi upload lại:
`python -m src.models.sources decontaminate && python -m src.models.sources manifest`

Vì sao `val ∩ test` là rủi ro nặng nhất: dù không train trên query đó, việc
chọn checkpoint/hyperparameter bằng val vẫn khiến metric test lạc quan hơn
thực tế. `data/benchmark_vi` **giữ nguyên** — decontamination nằm ở tầng
dataset của Method 2 nên bốn method vẫn được đánh giá trên cùng một tập test.


In [5]:
# Exit code != 0 → dừng notebook, không chạy tiếp cell training nào.
!python -m src.models.preflight \
    --config configs/method2/biencoder.yaml \
    --require-gpu T4 \
    --output results/method2/preflight.json

preflight = json.load(open('results/method2/preflight.json', encoding='utf-8'))
assert preflight['passed'], f"Preflight KHÔNG ĐẠT: {preflight['failures']}"
print('preflight PASS —', len(preflight['checks']), 'check')


[PASS] commit SHA — 6eb1b582b4ea (từ manifest, không có .git)
[PASS] working tree sạch — không áp dụng — chạy từ snapshot
[PASS] config parse được — configs/method2/biencoder.yaml (0180b9e53df282b7…)
[PASS] manifest tồn tại — data/method2/manifest.json
[PASS] decontamination.json tồn tại — data/method2/decontamination.json
[PASS] SHA-256 data/method2/decontamination.json — 37bf70a801f7f2cf…
[PASS] SHA-256 data/method2/tool_pool.json — 4b3357fa13adbd8c…
[PASS] SHA-256 data/method2/biencoder/train.jsonl — eaf94362480fe1d6…
[PASS] SHA-256 data/method2/crossencoder/train.jsonl — 24d0ef10e3724e01…
[PASS] overlap Bi-Encoder == 0 — {'test∩train': 0, 'test∩val': 0, 'train∩val': 0}
[PASS] overlap Cross-Encoder == 0 — {'test∩train': 0, 'test∩val': 0, 'train∩val': 0}
[PASS] unseen positive leakage == 0 — 0 tool
[PASS] hai stage dùng chung index — khớp
[PASS] version transformers — 5.15.1
[PASS] version sentence-transformers — 6.0.0
[PASS] version peft — 0.20.0
[PASS] version torch (ghi nhận) — 2.

In [6]:
# Số liệu split để đối chiếu bằng mắt trước khi tiêu giờ GPU.
stats = json.load(open('data/method2/biencoder/pairs_stats.json', encoding='utf-8'))
decon = stats['decontamination']

print('unique query/split :', stats['unique_queries_per_split'])
print('positive pairs     :', stats['n_positive_pairs'])
print('negative samples   :', stats['n_negative_samples'])
print('query trùng split  :', decon['n_overlapping_queries'], decon['overlapping_queries'])
print('sample bị loại     :', decon['rows_dropped_total'], decon['rows_dropped_by_transition'])
print('overlap còn lại    :', stats['split_overlap_after'])


unique query/split : {'test': 9331, 'train': 58829, 'val': 7819}
positive pairs     : 102100
negative samples   : 18334
query trùng split  : 1718 {'test∩train': 574, 'train∩val': 572, 'test∩train∩val': 542, 'test∩val': 30}
sample bị loại     : 32340 {'train->test': 26778, 'val->test': 3437, 'train->val': 2125}
overlap còn lại    : {'test∩train': 0, 'test∩val': 0, 'train∩val': 0}


# Phase 3 — Cross-Encoder (Kaggle T4)

Trích parameter theo định dạng BERT-QA: query là context, schema của
parameter là question. **Type routing lấy từ schema, không phải từ model** —
JSON được *lắp ráp* chứ không được *sinh ra*, nên không thể sai cú pháp và
không thể bịa tool name.

Cấu hình đúng plan §Phase 3, không cắt gì:

| | Giá trị | = plan |
|---|---|---|
| backbone | `xlm-roberta-base` (278M), **full fine-tune** | ✅ |
| `max_length` / padding | 256 / `longest` | ✅ |
| batch × grad_accum | 32 × 2 = **64 effective** | ✅ |
| `lr` / `head_lr` | 3e-5 / 1e-4 | ✅ |
| curriculum | warmup glaive+xLAM 2 epoch → finetune custom_vi 2 epoch @1e-5 | ✅ |

Khác Bi-Encoder ở một điểm quan trọng: vòng train **viết tay**, không dùng
`transformers.Trainer` (batch mang `schema_type` dạng `list[str]` để route
loss). Nên nó chạy thẳng trên `cuda:0` — **không có DataParallel**, không
cần `--single-gpu`, và thảm hoạ 475 s/step của Bi-Encoder không lặp lại.

Số lượng: warmup 130,068 cặp × 2 epoch + finetune 12,253 cặp × 2 epoch.


## Kiểm tra nhãn — KHÔNG sinh lại dataset

`data/method2/crossencoder/*.jsonl` đã được build ở local và **khoá SHA-256
trong `manifest.json`**. Chạy lại `crossencoder.dataset` ở đây sẽ:

1. đổi nội dung file → preflight fail vì hash lệch;
2. chết giữa chừng, vì `benchmark_vi/train.jsonl` cố tình **không** nằm
   trong dataset upload (chỉ có `test.jsonl`).

Sinh lại là lệnh preprocessing chạy ở local rồi upload lại, đúng như
`decontamination.json`. Ở đây chỉ đọc `label_stats.json` và kiểm gate §1.5.


In [7]:
stats = json.load(open('data/method2/label_stats.json', encoding='utf-8'))
print('SKIP rate      :', stats['skip_rate'])
print('theo lý do     :', stats['skip_by_reason'])
print('theo nguồn     :', stats['skip_rate_by_source'])
print('unsupported    :', stats['unsupported_type_coverage'])
print('kept theo type :', stats['kept_by_routing_type'])
print('has_value      :', stats['has_value_distribution'])
print('rows/split     :', stats['rows_per_split'])
# Gate §1.5: >30% thì span thuần không đủ, phải xem lại Q2 (fuzzy alignment).
assert stats['skip_rate'] <= 0.30, 'SKIP quá ngưỡng — xem lại Q2 trước khi train'


SKIP rate      : 0.2674
theo lý do     : {'non_verbatim': 49843, 'unsupported_type': 13441, 'required_without_value': 3836, 'type_mismatch': 9, 'enum_value_not_in_schema': 3}
theo nguồn     : {'xlam': 0.2753, 'glaive': 0.2677, 'custom_vi': 0.1884}
unsupported    : {'n_pairs': 13441, 'rate': 0.0535}
kept theo type : {'string': 104983, 'number': 69839, 'boolean': 5881, 'enum': 3197}
has_value      : {'positive': 146989, 'negative': 36911, 'negative_rate': 0.2007}
rows/split     : {'train': 142321, 'val': 17769, 'test': 23810}


## Smoke 50 step — đo trước khi tiêu giờ

Bài học từ Bi-Encoder: plan dự toán 4.5 h, thực tế lệch tới 8×. Đừng tin
dự toán, đo 50 step rồi ngoại suy.

Smoke ghi vào `smoke_run01/` riêng, tắt eval (17,769 cặp val mất vài phút,
lấn át số đo), save ở giữa để kiểm tra checkpoint ghi được. Mất ~3 phút.

`estimated_hours_full_run` là con số cần nhìn: **> 9 h thì dừng lại** và
cắt (giảm epoch warm-up xuống 1, hoặc `max_length` 256 → 192).


In [8]:
import subprocess, sys

CE = '/kaggle/working/artifacts/method2/crossencoder'
RUN_SMOKE = True

if RUN_SMOKE:
    subprocess.run(
        [sys.executable, '-m', 'src.models.crossencoder.train',
         '--config', 'configs/method2/crossencoder.yaml', '--smoke', '50'],
        check=True,
    )
    smoke = json.load(open(f'{CE}/smoke_run01/train_report.json', encoding='utf-8'))
    print()
    print('dựng dataset (s) :', smoke['dataset_build_seconds'])
    print('s/step           :', smoke['sec_per_step'])
    print('peak VRAM MB     :', smoke['peak_vram_mb'])
    print('cặp bỏ vì span ngoài cửa sổ:', smoke['n_dropped_unalignable'])
    print('optimizer step cả run      :', smoke['planned_optimizer_steps'])
    print('=> ƯỚC TÍNH CẢ RUN:', smoke['estimated_hours_full_run'], 'giờ')
    if (smoke['estimated_hours_full_run'] or 0) > 9:
        print()
        print('QUÁ NGÂN SÁCH — cắt trước khi chạy tiếp:')
        print('  curriculum.stages[0].epochs 2 -> 1  (tiết kiệm ~1/2)')
        print('  data.max_length 256 -> 192          (tiết kiệm ~25%)')
    # 2 checkpoint của smoke ~6.2 GB (xlm-r base full FT: trọng số 1.04 GB +
    # AdamW 2.07 GB mỗi bản). /kaggle/working chỉ có 20 GB mà run thật cần
    # ~13.4 GB — không xoá là hết đĩa giữa chừng.
    import shutil

    shutil.rmtree(f'{CE}/smoke_run01', ignore_errors=True)
    print('đã xoá smoke_run01 để lấy lại chỗ trống')
else:
    print('BỎ QUA smoke.')


[crossencoder] SMOKE: 50 step, output artifacts/method2/crossencoder/smoke_run01
[crossencoder] dựng dataset mất 30.0 s
[crossencoder] train=142318 val=17769 (bỏ 3 cặp không căn được span trong cửa sổ 256 token)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 400.61it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[crossencoder] giai đoạn warmup: 130065 sample, 4066 step, epoch 0..1
/kaggle/working/src/models/crossencoder/train.py:428: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.

{}

dựng dataset (s) : 30.0
s/step           : 1.44
peak VRAM MB     : 6822.8
cặp bỏ vì span ngoài cửa sổ: 3
optimizer step cả run      : 4450
=> ƯỚC TÍNH CẢ RUN: 1.78 giờ
đã xoá smoke_run01 để lấy lại chỗ trống


## Train — curriculum 2 giai đoạn, resume-safe

1. **Warm-up** glaive+xLAM, 2 epoch, lr 3e-5 — học kỹ năng span tổng quát.
2. **Fine-tune** custom_vi, 2 epoch, lr 1e-5 — domain đích.

Resume nạp lại **trọng số** (`from_pretrained`) chứ không chỉ optimizer
state, và bỏ qua đúng những giai đoạn/epoch đã xong — vị trí nằm trong
`progress.json` của checkpoint. Đứt session thì chạy lại cell này là đủ.


In [9]:
RUN = f'{CE}/run01'

# find_last_checkpoint trong train.py đã tự dò, kể cả bản gắn tag
# `stage-warmup`. Ở đây chỉ in ra cho thấy nó sẽ resume từ đâu.
from src.models.crossencoder.train import find_last_checkpoint

resume = find_last_checkpoint(RUN)
print('resume from:', resume or '(chưa có checkpoint — train từ đầu)')

subprocess.run(
    [sys.executable, '-m', 'src.models.crossencoder.train',
     '--config', 'configs/method2/crossencoder.yaml', '--output-dir', RUN],
    check=True,
)


resume from: (chưa có checkpoint — train từ đầu)


[crossencoder] dựng dataset mất 29.0 s
[crossencoder] train=142318 val=17769 (bỏ 3 cặp không căn được span trong cửa sổ 256 token)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2675.11it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[crossencoder] giai đoạn warmup: 130065 sample, 4066 step, epoch 0..1
/kaggle/working/src/models/crossencoder/train.py:428: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` befo

{
  "has_value_precision": 0.9737,
  "has_value_recall": 0.9739,
  "has_value_f1": 0.9738,
  "has_value_accuracy": 0.9579,
  "span_em": 0.8094,
  "span_start_accuracy": 0.8408,
  "enum_accuracy": 0.7454,
  "boolean_accuracy": 0.929,
  "n_span": 13685,
  "n_enum": 271,
  "n_boolean": 324,
  "loss": 1.1904
}


CompletedProcess(args=['/usr/bin/python3', '-m', 'src.models.crossencoder.train', '--config', 'configs/method2/crossencoder.yaml', '--output-dir', '/kaggle/working/artifacts/method2/crossencoder/run01'], returncode=0)

## Gate để qua Phase 4 (chế độ oracle retrieval)

| Metric | Ngưỡng |
|---|---|
| `has_value` F1 | ≥ 0.90 |
| Span EM | ≥ 0.80 |
| Enum accuracy | ≥ 0.90 |
| Boolean accuracy | ≥ 0.85 |

Đo trên **`custom_vi`**, không phải `overall` — plan §Phase 3 chốt gate ở
custom val, mà `overall` bị xLAM chi phối (14,452/17,769 cặp val).

Không assert cứng: gate là điểm quyết định, không phải lỗi. Trượt thì
artefact vẫn đáng giữ — cell lưu artifact ở dưới vẫn phải chạy.


In [10]:
!python -m src.models.crossencoder.evaluate \
    --config configs/method2/crossencoder.yaml \
    --model {RUN}/final \
    --pairs data/method2/crossencoder/val.jsonl \
    --output results/method2/metrics/crossencoder_val.json

ce = json.load(open('results/method2/metrics/crossencoder_val.json', encoding='utf-8'))
print('gate đo trên:', ce['gates']['measured_on'])
print()
print(f"{'slice':12s} {'has_value F1':>13s} {'span EM':>9s} {'enum':>7s} {'bool':>7s}")
for name in ['overall'] + sorted(ce['by_source']):
    m = ce['overall'] if name == 'overall' else ce['by_source'][name]
    print(f"{name:12s} {m['has_value_f1']:13.4f} {m['span_em']:9.4f}",
          f"{m['enum_accuracy']:7.4f} {m['boolean_accuracy']:7.4f}")
print()
if ce['gates']['passed']:
    print('GATE ĐẠT — sang được Phase 4')
else:
    print('GATE CHƯA ĐẠT:')
    for f in ce['gates']['failures']:
        print(f"  {f['metric']}: {f['value']} < {f['required']}")
    print('Thứ tự xử lý (plan §Phase 3): (a) thêm epoch finetune,')
    print('(b) tăng head_lr, (c) đổi sang multilingual-e5-base.')


Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 2530.04it/s]
{
  "overall": {
    "has_value_precision": 0.9735,
    "has_value_recall": 0.9739,
    "has_value_f1": 0.9737,
    "has_value_accuracy": 0.9577,
    "span_em": 0.8095,
    "span_start_accuracy": 0.8408,
    "enum_accuracy": 0.7454,
    "boolean_accuracy": 0.929,
    "n_span": 13685,
    "n_enum": 271,
    "n_boolean": 324
  },
  "by_source": {
    "custom_vi": {
      "has_value_precision": 0.9719,
      "has_value_recall": 0.9658,
      "has_value_f1": 0.9688,
      "has_value_accuracy": 0.9519,
      "span_em": 0.9339,
      "span_start_accuracy": 0.9624,
      "enum_accuracy": 0.7571,
      "boolean_accuracy": 1.0,
      "n_span": 772,
      "n_enum": 247,
      "n_boolean": 91
    },
    "glaive": {
      "has_value_precision": 0.991,
      "has_value_recall": 0.9904,
      "has_value_f1": 0.9907,
      "has_value_accuracy": 0.9825,
      "span_em": 0.9673,
      "span_start_accuracy": 0.973,
      "e

## Run manifest — audit

Cùng ràng buộc như Bi-Encoder: train xong mà không audit được thì coi như
chưa train. `argument_em` không có ở tầng thành phần (nó là metric
per-call, tính ở Phase 5 qua `src/evaluation`) nên gate đó để lại Phase 5.


In [11]:
!python -m src.models.run_manifest \
    --run-dir {RUN} \
    --config configs/method2/crossencoder.yaml \
    --stage crossencoder

manifest = json.load(open(f'{RUN}/run_manifest.json', encoding='utf-8'))
# `retrieval_metrics` chỉ áp dụng cho Bi-Encoder — bỏ khỏi danh sách bắt buộc.
missing = [m for m in manifest['audit_complete']['missing'] if m != 'retrieval_metrics']
print('thiếu           :', missing or 'không thiếu mục nào')
print('thời lượng (giờ):', manifest['train']['duration_hours'])
print('VRAM peak MB    :', manifest['train']['peak_vram_mb'])
print('checkpoint      :', manifest['train']['final_checkpoint'])
print('chọn checkpoint :', manifest['train']['checkpoint_selection'])
assert not missing, f'Chưa đủ artifact để audit: {missing}'


[run_manifest] → /kaggle/working/artifacts/method2/crossencoder/run01/run_manifest.json
[run_manifest] commit=6eb1b582b4eaba9ff8a9b51c4d88ae5bd0898465 dirty=False
[run_manifest] overlap sau decontamination: {'test∩train': 0, 'test∩val': 0, 'train∩val': 0}
[run_manifest] CHƯA ĐỦ: ['retrieval_metrics']
thiếu           : không thiếu mục nào
thời lượng (giờ): 0.667
VRAM peak MB    : 7703.3
checkpoint      : /kaggle/working/artifacts/method2/crossencoder/run01/final
chọn checkpoint : {'metric': 'has_value_f1', 'best_value': 0.9738, 'best_step': 4446, 'best_stage': 'finetune', 'n_evaluations': 10, 'note': 'Chỉ báo cáo — model cuối là checkpoint cuối, không nạp lại best.'}


In [12]:
# ===== Dọn dẹp + lưu artifact =====
# `trainer_state.pt` (2.07 GB/checkpoint) chỉ dùng để resume; train xong thì
# bỏ. Giữ lại trọng số của từng giai đoạn để còn ablation warm-up-only.
import os

freed = 0
for state in glob.glob(f'{RUN}/*/trainer_state.pt'):
    freed += os.path.getsize(state)
    os.remove(state)
print(f'giải phóng {freed / 1024**3:.1f} GB optimizer state')
!du -sh {RUN}/*

# Chỉ gói `final/` + metric: đó là thứ Phase 5 cần. Các checkpoint
# `stage-*` (mỗi bản ~1 GB) vẫn nằm nguyên trong Output của version nếu
# sau này cần ablation warm-up-only — nén chúng vào đây chỉ tốn 10 phút.
!tar czf /kaggle/working/crossencoder_run.tar.gz \
    -C /kaggle/working artifacts/method2/crossencoder/run01/final results/method2
!du -h /kaggle/working/crossencoder_run.tar.gz


giải phóng 8.3 GB optimizer state
1.1G	/kaggle/working/artifacts/method2/crossencoder/run01/checkpoint-3500
1.1G	/kaggle/working/artifacts/method2/crossencoder/run01/checkpoint-4000
1.1G	/kaggle/working/artifacts/method2/crossencoder/run01/final
16K	/kaggle/working/artifacts/method2/crossencoder/run01/run_manifest.json
1.1G	/kaggle/working/artifacts/method2/crossencoder/run01/stage-finetune
1.1G	/kaggle/working/artifacts/method2/crossencoder/run01/stage-warmup
8.0K	/kaggle/working/artifacts/method2/crossencoder/run01/train_report.json
787M	/kaggle/working/crossencoder_run.tar.gz
